# nb11: Lightweight SISSO-style Symbolic Regression

**Goal:** Find an interpretable closed-form formula `alpha_R = f(physical features)` that approximates Rashba parameter, using a SISSO-inspired approach. Compare its accuracy to the C6 XGBoost baseline (R² = 0.643).

## Why bother

XGBoost gives you R² = 0.643 but no formula. SISSO gives you a 1D, 2D, or 3D analytical descriptor — a closed-form expression that goes directly into the DDP report and is defensible to physicists who don't trust black-box ML. Even if the SISSO R² is lower (likely 0.45-0.55 for 1D-2D), having both is a much stronger story than XGBoost alone.

## What SISSO does, in one paragraph

Take a small set of "primary features" (numeric quantities you trust). Apply a bag of operators (+, -, *, /, sqrt, log, exp, square, cube, inverse) to combine them into a vast pool of candidate "compound features" (e.g. `sqrt(f1) * f2 / f3`). This is "feature creation". Then apply Sure Independence Screening (rank by correlation with target, keep top K) followed by sparse linear regression (L0/L1 penalty) on the survivors. The output: 1-3 compound features whose linear combination predicts the target. Output is a closed-form analytical expression.

## Why we implement this lightweight (not pysisso)

`pysisso` requires the SISSO++ Fortran/C++ executable. Installing it on Vast.ai is a hassle. For a problem with **only 6 primary features and rung-2 compound features**, the search space is small enough (~hundreds to a few thousand candidates) that we can do feature creation explicitly in numpy and use sklearn `Lasso` for sparse selection. This captures the algorithmic essence of SISSO without the build dependency.

If your prof asks "is this real SISSO?" — the answer is "no, this is a lightweight implementation of the same idea (feature creation + sparse selection). For the final paper we should run pysisso and compare." Be honest about it.

## Inputs

- 6 primary features = the C6 set:
  `E_pfrac_VBM, E_pfrac_CBM, radius_mean, pmid_afs_gauss_std, kpath_angle_deg, ehull`
- Operators: `+ - * / sqrt log exp square cube inverse abs`
- **Rung 1**: single operator applied to one or two primary features
- **Rung 2**: operator applied to a rung-1 feature and a primary feature (compositions like `(f1 / f2) * f3`, `sqrt(f1 + f2)`)


## Cell 1: Imports & paths

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from itertools import combinations, product
from time import time

from sklearn.linear_model import Lasso, LassoCV, LinearRegression
from sklearn.model_selection import LeaveOneOut, cross_val_predict, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

BASE_DIR = os.path.abspath(os.path.join('..'))
OLD_CSV = os.path.join(BASE_DIR, 'k-path', 'old+new_nb6', 'rashba_206_all_descriptors_old.csv')
NEW_CSV = os.path.join(BASE_DIR, 'k-path', 'old+new_nb6', 'rashba_206_all_descriptors.csv')
RESULTS_DIR = os.path.join('.', 'nb11_sisso-results')
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Setup OK. Results dir:', RESULTS_DIR)


Setup OK. Results dir: .\nb11_sisso-results


## Cell 2: Load and collapse to 99 compounds

In [2]:
df_old = pd.read_csv(OLD_CSV)
df_new = pd.read_csv(NEW_CSV)
ID_COLS = ['Formula', 'uid', 'kpath', 'Rashba_parameter']
TARGET = 'Rashba_parameter'

df_merged = df_old[ID_COLS].copy()
old_features = [c for c in df_old.columns if c not in ID_COLS]
new_features = [c for c in df_new.columns if c not in ID_COLS]
overlap = set(old_features) & set(new_features)
for col in old_features:
    df_merged[f'old_{col}' if col in overlap else col] = df_old[col].values
for col in new_features:
    df_merged[f'new_{col}' if col in overlap else col] = df_new[col].values

idx_max = df_merged.groupby('uid')[TARGET].idxmax()
df_99 = df_merged.loc[idx_max].reset_index(drop=True)
y = df_99[TARGET].values
print(f'99-row df: {df_99.shape}')

PRIMARY_RAW = ['E_pfrac_VBM', 'E_pfrac_CBM', 'radius_mean',
               'pmid_afs_gauss_std', 'kpath_angle_deg', 'ehull']

def resolve(name):
    if name in df_99.columns: return name
    if f'old_{name}' in df_99.columns: return f'old_{name}'
    if f'new_{name}' in df_99.columns: return f'new_{name}'
    raise KeyError(name)

PRIMARY_COLS = [resolve(n) for n in PRIMARY_RAW]
print('Primary features (resolved):')
for raw, col in zip(PRIMARY_RAW, PRIMARY_COLS):
    print(f'  {raw:25s} -> {col}')

# Pull primary feature matrix; clean NaN/inf
X_prim = df_99[PRIMARY_COLS].copy()
X_prim = X_prim.fillna(X_prim.median())
print(f'\nPrimary matrix: {X_prim.shape}')
print(X_prim.describe().round(3))


99-row df: (99, 1393)
Primary features (resolved):
  E_pfrac_VBM               -> old_E_pfrac_VBM
  E_pfrac_CBM               -> old_E_pfrac_CBM
  radius_mean               -> old_radius_mean
  pmid_afs_gauss_std        -> pmid_afs_gauss_std
  kpath_angle_deg           -> old_kpath_angle_deg
  ehull                     -> old_ehull

Primary matrix: (99, 6)
       old_E_pfrac_VBM  old_E_pfrac_CBM  old_radius_mean  pmid_afs_gauss_std  \
count           99.000           99.000           99.000              99.000   
mean             0.645            0.550            1.319               0.868   
std              0.308            0.361            0.093               1.169   
min              0.134            0.093            1.100               0.000   
25%              0.243            0.149            1.250               0.003   
50%              0.823            0.772            1.317               0.013   
75%              0.894            0.902            1.383               1.447   
m

## Cell 3: Feature creation — Rung 1

Apply unary operators to each primary feature, and binary operators to each pair.

**Critical:** when we apply `log` or `sqrt` or `1/x`, we have to handle non-positive or zero values. Strategy:
- `sqrt(x)` only if `x >= 0` for all entries
- `log(x)` only if `x > 0` for all entries (else skip)
- `1/x` only if `x` is never zero (else skip)
- Never apply `exp` to large values (overflow); cap at exp(min(x, 50))

Each created feature is checked for finite values; any feature with NaN/inf entries is dropped.

In [3]:
def safe_unary(name, x, op):
    """Apply unary op safely. Returns (new_name, new_array) or None if invalid."""
    x = np.asarray(x, dtype=float)
    try:
        if op == 'sqrt':
            if (x < 0).any(): return None
            return f'sqrt({name})', np.sqrt(x)
        if op == 'log':
            if (x <= 0).any(): return None
            return f'log({name})', np.log(x)
        if op == 'exp':
            # Avoid overflow
            if (x > 50).any() or (x < -50).any(): return None
            return f'exp({name})', np.exp(x)
        if op == 'square':
            return f'({name})^2', x ** 2
        if op == 'cube':
            return f'({name})^3', x ** 3
        if op == 'inverse':
            if (x == 0).any() or (np.abs(x) < 1e-10).any(): return None
            return f'1/({name})', 1.0 / x
        if op == 'abs':
            return f'abs({name})', np.abs(x)
    except (FloatingPointError, OverflowError):
        return None
    return None


def safe_binary(n1, x1, n2, x2, op):
    """Apply binary op. Returns (new_name, new_array) or None."""
    x1 = np.asarray(x1, dtype=float); x2 = np.asarray(x2, dtype=float)
    try:
        if op == '+': return f'({n1} + {n2})', x1 + x2
        if op == '-': return f'({n1} - {n2})', x1 - x2
        if op == '*': return f'({n1} * {n2})', x1 * x2
        if op == '/':
            if (x2 == 0).any() or (np.abs(x2) < 1e-10).any(): return None
            return f'({n1} / {n2})', x1 / x2
    except Exception:
        return None
    return None


def is_finite_and_varying(arr):
    if not np.all(np.isfinite(arr)): return False
    if np.std(arr) < 1e-10: return False    # constant
    return True


# Build rung 1
unary_ops = ['sqrt', 'log', 'square', 'cube', 'inverse', 'abs']   # skip exp by default (rare it helps)
binary_ops = ['+', '-', '*', '/']

rung1 = {}
# Identity (primary features are themselves rung-1)
for col in PRIMARY_COLS:
    rung1[col] = X_prim[col].values

# Unary
for col in PRIMARY_COLS:
    for op in unary_ops:
        result = safe_unary(col, X_prim[col].values, op)
        if result and is_finite_and_varying(result[1]):
            rung1[result[0]] = result[1]

# Binary
for c1, c2 in combinations(PRIMARY_COLS, 2):
    for op in binary_ops:
        result = safe_binary(c1, X_prim[c1].values, c2, X_prim[c2].values, op)
        if result and is_finite_and_varying(result[1]):
            rung1[result[0]] = result[1]
        # Non-commutative: also try reversed for - and /
        if op in ['-', '/']:
            result = safe_binary(c2, X_prim[c2].values, c1, X_prim[c1].values, op)
            if result and is_finite_and_varying(result[1]):
                rung1[result[0]] = result[1]

print(f'Rung 1 features: {len(rung1)}')
print('Sample:', list(rung1.keys())[:8])


Rung 1 features: 110
Sample: ['old_E_pfrac_VBM', 'old_E_pfrac_CBM', 'old_radius_mean', 'pmid_afs_gauss_std', 'old_kpath_angle_deg', 'old_ehull', 'sqrt(old_E_pfrac_VBM)', 'log(old_E_pfrac_VBM)']


## Cell 4: Feature creation — Rung 2

Apply unary operators to rung-1 features (already done implicitly for the primary subset), and binary operators between rung-1 features.

**Combinatorial blowup:** rung-1 has ~50 features. Pairs = ~1000. Times 4 binary ops × 2 for non-commutative = ~8000 candidates. We sample / filter aggressively.

In [4]:
# Limit rung-2 search: only combine rung-1 features with high SD-correlation
# (This is the "Sure Independence Screening" step from SISSO.)
# We keep top K rung-1 features by absolute correlation with target.

K_SCREEN = 20  # keep top 20 rung-1 by |correlation with target|
corrs = {name: abs(np.corrcoef(arr, y)[0, 1]) for name, arr in rung1.items()
         if not np.isnan(np.corrcoef(arr, y)[0, 1])}
top_rung1 = sorted(corrs.items(), key=lambda t: -t[1])[:K_SCREEN]
print(f'Top {K_SCREEN} rung-1 features by |corr with target|:')
for name, c in top_rung1[:10]:
    print(f'  |r|={c:.3f}  {name}')

# Build rung 2 from these
top_names = [n for n, _ in top_rung1]
rung2 = dict(rung1)   # rung 2 includes all rung 1

for c1 in top_names:
    for op in unary_ops:
        result = safe_unary(c1, rung1[c1], op)
        if result and is_finite_and_varying(result[1]) and result[0] not in rung2:
            rung2[result[0]] = result[1]

for c1, c2 in combinations(top_names, 2):
    for op in binary_ops:
        result = safe_binary(c1, rung1[c1], c2, rung1[c2], op)
        if result and is_finite_and_varying(result[1]) and result[0] not in rung2:
            rung2[result[0]] = result[1]
        if op in ['-', '/']:
            result = safe_binary(c2, rung1[c2], c1, rung1[c1], op)
            if result and is_finite_and_varying(result[1]) and result[0] not in rung2:
                rung2[result[0]] = result[1]

print(f'\nRung 2 features (cumulative): {len(rung2)}')


Top 20 rung-1 features by |corr with target|:
  |r|=0.325  (old_E_pfrac_CBM / old_E_pfrac_VBM)
  |r|=0.294  (old_E_pfrac_VBM * old_ehull)
  |r|=0.290  (old_E_pfrac_VBM - old_E_pfrac_CBM)
  |r|=0.290  (old_E_pfrac_CBM - old_E_pfrac_VBM)
  |r|=0.278  (old_radius_mean + old_ehull)
  |r|=0.256  1/(old_radius_mean)
  |r|=0.243  log(old_radius_mean)
  |r|=0.236  sqrt(old_radius_mean)
  |r|=0.229  (pmid_afs_gauss_std / old_E_pfrac_CBM)
  |r|=0.228  old_radius_mean

Rung 2 features (cumulative): 1213


## Cell 5: 1D descriptor — best single feature

The simplest SISSO output: find the single compound feature (from rung 2) that best linearly predicts the target.

In [5]:
# Compute correlation of every rung-2 feature with target
rung2_names = list(rung2.keys())
rung2_corrs = {name: abs(np.corrcoef(rung2[name], y)[0, 1])
               for name in rung2_names
               if not np.isnan(np.corrcoef(rung2[name], y)[0, 1])}
ranked = sorted(rung2_corrs.items(), key=lambda t: -t[1])
print('Top 15 rung-2 features by |Pearson correlation| with target:')
for name, c in ranked[:15]:
    print(f'  |r|={c:.3f}  {name}')

# Best 1D descriptor: fit linear regression with that single feature
best_1d_name = ranked[0][0]
x_1d = rung2[best_1d_name].reshape(-1, 1)

lr = LinearRegression()
loo = LeaveOneOut()
y_pred = cross_val_predict(lr, x_1d, y, cv=loo)
r2_1d = r2_score(y, y_pred)
mae_1d = mean_absolute_error(y, y_pred)
lr.fit(x_1d, y)

print(f'\n1D descriptor: {best_1d_name}')
print(f'  Formula: alpha_R = {lr.intercept_:.4f} + {lr.coef_[0]:.4f} * [{best_1d_name}]')
print(f'  LOO R2 = {r2_1d:.4f}, MAE = {mae_1d:.4f}')


Top 15 rung-2 features by |Pearson correlation| with target:
  |r|=0.437  ((old_E_pfrac_VBM - old_E_pfrac_CBM) / (old_E_pfrac_VBM / old_E_pfrac_CBM))
  |r|=0.437  ((old_E_pfrac_CBM - old_E_pfrac_VBM) / (old_E_pfrac_VBM / old_E_pfrac_CBM))
  |r|=0.437  ((old_E_pfrac_CBM / old_E_pfrac_VBM) * (old_E_pfrac_VBM - old_E_pfrac_CBM))
  |r|=0.437  ((old_E_pfrac_CBM / old_E_pfrac_VBM) * (old_E_pfrac_CBM - old_E_pfrac_VBM))
  |r|=0.396  ((old_E_pfrac_CBM / old_E_pfrac_VBM))^3
  |r|=0.392  ((old_E_pfrac_VBM - old_E_pfrac_CBM) / (old_radius_mean / old_E_pfrac_CBM))
  |r|=0.392  ((old_E_pfrac_CBM - old_E_pfrac_VBM) / (old_radius_mean / old_E_pfrac_CBM))
  |r|=0.391  ((old_E_pfrac_VBM - old_E_pfrac_CBM) / 1/(old_E_pfrac_CBM))
  |r|=0.391  ((old_E_pfrac_CBM - old_E_pfrac_VBM) / 1/(old_E_pfrac_CBM))
  |r|=0.388  ((old_E_pfrac_CBM / old_E_pfrac_VBM) / (old_E_pfrac_VBM + pmid_afs_gauss_std))
  |r|=0.380  ((old_E_pfrac_CBM / old_E_pfrac_VBM) / log(old_radius_mean))
  |r|=0.372  ((old_E_pfrac_CBM / old_E_p

## Cell 6: 2D descriptor — Lasso on top-K rung-2 features

Take the top 50 rung-2 features by correlation, standardize, run LassoCV. The 2 (or 3) features with non-zero coefficients become the 2D descriptor.

In [6]:
K = 50  # top K rung-2 features for sparse regression
top_K = [name for name, _ in ranked[:K]]
X_lasso = np.column_stack([rung2[n] for n in top_K])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_lasso)

# LassoCV picks alpha by CV
lasso = LassoCV(cv=10, random_state=42, max_iter=20000, n_alphas=100)
lasso.fit(X_scaled, y)

# Sparse coefficients
nonzero_idx = np.where(np.abs(lasso.coef_) > 1e-6)[0]
print(f'LassoCV picked {len(nonzero_idx)} non-zero features (alpha = {lasso.alpha_:.4f})')

# Rank chosen features by |coefficient|
chosen = sorted([(top_K[i], lasso.coef_[i]) for i in nonzero_idx],
                key=lambda t: -abs(t[1]))
print('\nChosen features (sorted by |coef|):')
for name, c in chosen[:10]:
    print(f'  {c:+.4f}  *  {name}')

# Evaluate the 2D model: take top 2 chosen features, refit linear regression
if len(chosen) >= 2:
    feats_2d = [chosen[0][0], chosen[1][0]]
    X_2d = np.column_stack([rung2[n] for n in feats_2d])
    lr = LinearRegression()
    y_pred = cross_val_predict(lr, X_2d, y, cv=LeaveOneOut())
    r2_2d = r2_score(y, y_pred)
    mae_2d = mean_absolute_error(y, y_pred)
    lr.fit(X_2d, y)
    print(f'\n2D descriptor:')
    for name, coef in zip(feats_2d, lr.coef_):
        print(f'  {coef:+.4f}  *  [{name}]')
    print(f'  + {lr.intercept_:.4f}')
    print(f'  LOO R2 = {r2_2d:.4f}, MAE = {mae_2d:.4f}')
else:
    print('Lasso picked < 2 features. Try lower alpha.')
    feats_2d = []
    r2_2d = np.nan


LassoCV picked 5 non-zero features (alpha = 0.0448)

Chosen features (sorted by |coef|):
  -0.2164  *  ((old_E_pfrac_VBM - old_E_pfrac_CBM) / (old_E_pfrac_VBM / old_E_pfrac_CBM))
  +0.1909  *  (1/(old_radius_mean) / (old_E_pfrac_VBM + pmid_afs_gauss_std))
  -0.1111  *  ((old_E_pfrac_VBM * old_ehull) + sqrt(old_radius_mean))
  +0.0399  *  ((old_E_pfrac_CBM / old_E_pfrac_VBM))^3
  -0.0389  *  ((old_E_pfrac_VBM * old_ehull) - 1/(old_radius_mean))

2D descriptor:
  -5.2062  *  [((old_E_pfrac_VBM - old_E_pfrac_CBM) / (old_E_pfrac_VBM / old_E_pfrac_CBM))]
  +0.3997  *  [(1/(old_radius_mean) / (old_E_pfrac_VBM + pmid_afs_gauss_std))]
  + 1.5886
  LOO R2 = 0.1892, MAE = 0.6440


## Cell 7: 3D descriptor

Same as 2D but using the top 3 chosen features.

In [7]:
if len(chosen) >= 3:
    feats_3d = [chosen[0][0], chosen[1][0], chosen[2][0]]
    X_3d = np.column_stack([rung2[n] for n in feats_3d])
    lr = LinearRegression()
    y_pred = cross_val_predict(lr, X_3d, y, cv=LeaveOneOut())
    r2_3d = r2_score(y, y_pred)
    mae_3d = mean_absolute_error(y, y_pred)
    lr.fit(X_3d, y)
    print('3D descriptor:')
    for name, coef in zip(feats_3d, lr.coef_):
        print(f'  {coef:+.4f}  *  [{name}]')
    print(f'  + {lr.intercept_:.4f}')
    print(f'  LOO R2 = {r2_3d:.4f}, MAE = {mae_3d:.4f}')
else:
    print('Lasso did not pick 3+ features. Try lower alpha.')
    feats_3d = []
    r2_3d = np.nan


3D descriptor:
  -4.0579  *  [((old_E_pfrac_VBM - old_E_pfrac_CBM) / (old_E_pfrac_VBM / old_E_pfrac_CBM))]
  +0.3977  *  [(1/(old_radius_mean) / (old_E_pfrac_VBM + pmid_afs_gauss_std))]
  -2.1084  *  [((old_E_pfrac_VBM * old_ehull) + sqrt(old_radius_mean))]
  + 4.1080
  LOO R2 = 0.1891, MAE = 0.6361


## Cell 8: Comparison summary

Compare 1D, 2D, 3D analytical descriptors against the C6 XGBoost baseline.

In [8]:
print('=' * 70)
print('  nb11 SISSO-LITE SUMMARY')
print('=' * 70)
print(f'C6 XGBoost baseline (from nb7):   R2 = 0.643')
print(f'1D analytical descriptor:         R2 = {r2_1d:.4f}')
if not np.isnan(r2_2d):
    print(f'2D analytical descriptor:         R2 = {r2_2d:.4f}')
if not np.isnan(r2_3d):
    print(f'3D analytical descriptor:         R2 = {r2_3d:.4f}')

print()
print('Interpretation:')
print('  - If 2D/3D descriptors come within ~0.05 of C6, you have an interpretable')
print('    formula that nearly matches the black-box model. STRONG result for the report.')
print('  - If they fall short by >0.10, the relationship is genuinely non-linear and')
print('    XGBoost is doing real work the formula cannot capture. Still worth reporting.')

# Save the final formulas to a text file
with open(os.path.join(RESULTS_DIR, 'descriptors.txt'), 'w') as f:
    f.write('# Analytical descriptors found by lightweight SISSO\n\n')
    f.write(f'1D: R2 = {r2_1d:.4f}\n')
    f.write(f'    alpha_R ~ {best_1d_name}\n\n')
    if feats_2d:
        f.write(f'2D: R2 = {r2_2d:.4f}\n')
        f.write(f'    Features: {feats_2d}\n\n')
    if feats_3d:
        f.write(f'3D: R2 = {r2_3d:.4f}\n')
        f.write(f'    Features: {feats_3d}\n')
print(f'\nSaved: {os.path.join(RESULTS_DIR, "descriptors.txt")}')


  nb11 SISSO-LITE SUMMARY
C6 XGBoost baseline (from nb7):   R2 = 0.643
1D analytical descriptor:         R2 = 0.1531
2D analytical descriptor:         R2 = 0.1892
3D analytical descriptor:         R2 = 0.1891

Interpretation:
  - If 2D/3D descriptors come within ~0.05 of C6, you have an interpretable
    formula that nearly matches the black-box model. STRONG result for the report.
  - If they fall short by >0.10, the relationship is genuinely non-linear and
    XGBoost is doing real work the formula cannot capture. Still worth reporting.

Saved: .\nb11_sisso-results\descriptors.txt


## Notes for next steps

1. **This is not full SISSO.** Real SISSO does L0-regularized search over the full compound feature space (not Lasso = L1 + screening). For the paper, run actual `pysisso` and compare results — the formulas may be similar but the implementation should be cited correctly.

2. **Rung 3 is feasible.** With 6 primary features and our screening, rung-3 search would be ~50,000 candidates and takes a few minutes. Add a Cell 4b if you want to push deeper.

3. **Adding asymmetry features as primary.** If nb10's ASI features come out strong, add them to `PRIMARY_COLS` here and re-run. SISSO works better when primary features have direct physical meaning.

4. **Verify on held-out.** As with nb9 and nb10, the LOO score here is selection-biased. Final paper number should be from nested CV or a held-out test set.
